# PaleographIA — participação da equipe no Google Colab

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aj1no/paleographia-htr/blob/main/docs/entregas/pi_iv_pln_20261023_colab/colaboracao_colab.ipynb)

1. Abra pelo botão e escolha **Arquivo → Salvar uma cópia no Drive**. Use seu nome no título.
2. Conecte ao ambiente padrão de **CPU** e use **Ambiente de execução → Executar tudo**.
   A primeira célula busca os dados e prepara as bibliotecas. Aguarde a conclusão.
3. Na seção com seu nome, leia a tarefa e preencha os cinco campos. A data deve ser
   a data real da atividade, no formato AAAA-MM-DD. Separe evidências por ponto e vírgula.
   Execute novamente **a célula do seu formulário** para salvar o relato.
4. Ao final, em **Baixar minha contribuição**, selecione seu nome, marque
   `baixar_contribuicoes` e execute essa célula. Envie o **ZIP baixado** à coordenação
   pelo grupo. Salve também sua cópia do notebook no Drive.

**Salvamento:** os JSONs ficam na máquina temporária do Colab. Salvar a cópia do
notebook no Drive não salva esses arquivos. Baixe o ZIP antes de encerrar a sessão.
O ZIP leva os relatos em JSON; evidências externas e arquivos produzidos por você
devem ser enviados junto, quando necessários. Nenhum envio ao grupo ou ao GitHub
é feito automaticamente. Sem formulário preenchido, nenhum relato é criado.

Cada participante descreve o que realmente discutiu, analisou ou revisou e informa
o uso de IA. Os relatos continuam sujeitos à conferência. Não é preciso carregar
imagens, montar o Drive, baixar pesos ou habilitar GPU para esta atividade.

Os dados vêm da revisão `c8a3e9a7f1cc55b66d630c510eff17ff95dfdece`, para manter o diagnóstico
reproduzível. A preparação exige internet. Para reler as instruções no GitHub,
consulte o [guia do Colab](https://github.com/aj1no/paleographia-htr/blob/main/docs/entregas/pi_iv_pln_20261023_colab/README.md).


In [ ]:
# @title 1. Preparar o projeto (execute primeiro)
"""Preparação autocontida do notebook Colab; incorporada à primeira célula."""
from pathlib import Path
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys

REPOSITORIO_COLAB = 'https://github.com/aj1no/paleographia-htr.git'
REVISAO_DADOS_COLAB = 'c8a3e9a7f1cc55b66d630c510eff17ff95dfdece'
PASTAS_COLAB = [
    'src',
    'docs/experimentos/traslado_269r_20261016/support',
    'docs/experimentos/traslado_269r_reconhecedores_24_20261020',
    'docs/revisao_humana/traslado_269r/images',
    'docs/revisao_humana/traslado_269r/referencia_assistida_20261020',
    'docs/revisao_humana/traslado_269r/reconferencia_ressalvas_20261021',
    'docs/entregas/pi_iv_pln_20261020',
    'docs/entregas/pi_iv_pln_20261022',
    'docs/entregas/pi_iv_pln_20261023_entrega',
    'docs/entregas/pi_iv_pln_20261023_equipe',
    'docs/bibliografia/leitura_t1_t4_20261023',
]
# O notebook usa os módulos de métricas diretamente de src/. Não instala a
# aplicação, seus extras de OCR ou versões de NumPy/Matplotlib sobre as do Colab.
REQUISITOS_COLAB = [
    'jiwer==4.0.0', 'Pillow>=10,<13', 'pydantic>=2.11.7,<3',
    'numpy>=1.26,<3', 'matplotlib>=3.8,<4', 'nbformat>=5.10,<6', 'nbclient>=0.10,<1',
]


def esta_no_colab():
    if 'google.colab' in sys.modules:
        return True
    try:
        return importlib.util.find_spec('google.colab') is not None
    except (ModuleNotFoundError, ValueError):
        return False


def executar_git(argumentos, cwd=None):
    result = subprocess.run(['git', *argumentos], cwd=cwd, check=True,
                            capture_output=True, text=True,
                            env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    return result.stdout.strip()


def preparar_checkout(destino):
    destino = Path(destino).resolve()
    if destino.exists():
        if not (destino / '.git').is_dir():
            raise RuntimeError('A pasta escolhida já existe e não é o checkout preparado. Use outra pasta.')
        remoto = executar_git(['remote', 'get-url', 'origin'], destino)
        revisao = executar_git(['rev-parse', 'HEAD'], destino)
        if remoto != REPOSITORIO_COLAB or revisao != REVISAO_DADOS_COLAB:
            raise RuntimeError('Checkout existente diferente do esperado; nenhuma alteração foi feita.')
        return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    executar_git(['clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
                  '--branch', 'main', REPOSITORIO_COLAB, str(destino)])
    executar_git(['sparse-checkout', 'init', '--cone'], destino)
    executar_git(['sparse-checkout', 'set', *PASTAS_COLAB], destino)
    executar_git(['fetch', '--depth', '1', '--filter=blob:none', 'origin', REVISAO_DADOS_COLAB], destino)
    executar_git(['checkout', '--detach', REVISAO_DADOS_COLAB], destino)
    if executar_git(['rev-parse', 'HEAD'], destino) != REVISAO_DADOS_COLAB:
        raise RuntimeError('Não foi possível conferir a revisão dos dados.')
    return destino


def instalar_dependencias():
    from packaging.requirements import Requirement
    pendentes = []
    for expressao in REQUISITOS_COLAB:
        requisito = Requirement(expressao)
        try:
            versao = importlib.metadata.version(requisito.name)
        except importlib.metadata.PackageNotFoundError:
            pendentes.append(expressao)
            continue
        if versao not in requisito.specifier:
            pendentes.append(expressao)
    if pendentes:
        print('Preparando bibliotecas do notebook:', ', '.join(pendentes))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *pendentes], check=True)
        importlib.invalidate_caches()
    return pendentes


def preparar_ambiente_colab(colab=None, destino=None, instalar=True):
    no_colab = esta_no_colab() if colab is None else colab
    if no_colab:
        raiz = preparar_checkout(destino or Path('/content/PaleographIA_equipe_c8a3e9a'))
        instalados = instalar_dependencias() if instalar else []
    else:
        inicio = Path.cwd().resolve()
        raiz = next((p for p in (inicio, *inicio.parents)
                     if (p / 'pyproject.toml').is_file() and (p / 'src/paleographia').is_dir()), None)
        if raiz is None:
            raise RuntimeError('Abra pelo botão do Colab ou execute dentro do checkout do PaleographIA.')
        instalados = []
    sys.dont_write_bytecode = True
    os.chdir(raiz)
    if str(raiz / 'src') not in sys.path:
        sys.path.insert(0, str(raiz / 'src'))
    saida = raiz / 'output/equipe_colab_20261023'
    saida.mkdir(parents=True, exist_ok=True)
    registro = dict(colab=no_colab, revisao_dados=REVISAO_DADOS_COLAB if no_colab else 'checkout_local',
                    bibliotecas_instaladas=instalados, python=sys.version,
                    novas_inferencias=0, montagem_drive=False)
    (saida / 'preparacao.json').write_text(json.dumps(registro, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
    print('Ambiente preparado. Execute as próximas células; use a seção com seu nome.')
    return raiz

RAIZ_COLAB = preparar_ambiente_colab()
NO_COLAB = esta_no_colab()


# PaleographIA — diagnóstico reproduzível e participação da equipe

**24 linhas do 269r; dois reconhecedores; referência assistida e oito ressalvas.**

Equipe: Caio Augusto de Souza Soares; Guilherme Rodrigues Veiga;
Rodolfo Vinicius Cima Takemoto; Tiago Galhardo Avelar.
FATEC Jundiaí, Ciência de Dados, 5º semestre, Jundiaí – SP.

## Como trabalhar em equipe neste notebook

1. Execute a parte comum para reproduzir e compreender o diagnóstico.
2. Vá à seção **Participação individual** e faça a atividade da sua frente.
3. Preencha seu formulário com o que você realmente discutiu, analisou ou revisou.
   Inclua a data real, a evidência e o uso de IA. Participação em discussão
   também pode ser registrada: explique a sua contribuição concreta.
4. Execute sua célula. Um formulário vazio não cria registro. Um preenchido
   gera um novo JSON em `output/participacao_equipe_20261023/`, para conferência.
5. Compartilhe esse JSON e as observações com a equipe. A execução do notebook
   por outra pessoa ou por IA não comprova sua participação individual.

**Situação relatada pelo solicitante em 23/10:** a equipe discutiu o projeto,
mas a execução prática está concentrada nele. Esse relato coletivo
não identifica retrospectivamente a tarefa feita por cada colega. A divisão do
artigo inicial é **planejamento**, adaptado abaixo ao experimento atual.

| Integrante | Frente prevista/adaptada | Registro novo nesta versão |
| --- | --- | --- |
| Caio | Bibliografia e interpretação das fontes | A preencher pelo participante |
| Guilherme | Modelos, ambiente e condições da comparação | A preencher pelo participante |
| Tiago | Inventário, associação imagem–texto e inspeção de erros | A preencher pelo participante |
| Rodolfo | Coordenação, dados e consolidação | A preencher; histórico de revisão do manuscrito preservado |

O [roteiro e cronograma](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/entregas/pi_iv_pln_20261023_equipe/roteiro_equipe.md) organiza a rodada. A seção histórica
de reconferência identifica os 17 registros de Rodolfo; ela não é redistribuída
entre autores. Discussão relatada, atividade planejada e execução documentada
continuam distintas.

## Escopo da execução

A parte comum deriva do notebook de 22/10 e recalcula o experimento de 20/10.
Não baixa pesos, não executa reconhecedores, não treina modelos e não altera
catálogo, editor, alvos ou revisões. Os dez códigos originais são mantidos,
com mudança apenas da pasta dos derivados. A execução de demonstração desta
versão é por Codex (IA), com os quatro formulários vazios.

O [artigo revisto](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/entregas/pi_iv_pln_20261023_entrega/README.md) e os [fichamentos integrais](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/bibliografia/leitura_t1_t4_20261023/README.md)
estão disponíveis. O treinamento técnico posterior e a expansão do corpus têm
evidências próprias e não mudam os resultados deste diagnóstico.


## 1. Ambiente e localização dos arquivos

Requer o projeto instalado e `../pi_iv_pln_20261022/requirements-notebook.txt`. Execute as células em ordem.
A localização funciona a partir da raiz do repositório ou da pasta deste notebook.
Os resultados derivados podem ser recriados; os pacotes históricos são somente lidos.

In [ ]:
import csv
import hashlib
import html
import importlib.metadata
import json
import platform
import runpy
import sys
from collections import Counter, defaultdict
from pathlib import Path

from IPython.display import HTML, Markdown, display

def localizar_raiz(inicio):
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'pyproject.toml').is_file() and (candidato / 'src/paleographia').is_dir():
            return candidato
    raise RuntimeError('Execute dentro do repositório PaleographIA.')

RAIZ = localizar_raiz(Path.cwd().resolve())
sys.path.insert(0, str(RAIZ / 'src'))
from paleographia.evaluation import aggregate, counts, normalize

PACOTE = RAIZ / 'docs/experimentos/traslado_269r_reconhecedores_24_20261020'
REFERENCIA = RAIZ / 'docs/revisao_humana/traslado_269r/referencia_assistida_20261020'
SAIDA = RAIZ / 'output/equipe_colab_20261023/calculos'
SAIDA.mkdir(parents=True, exist_ok=True)
(SAIDA / 'figuras').mkdir(exist_ok=True)
(SAIDA / 'tabelas').mkdir(exist_ok=True)

def ler_json(path):
    return json.loads(path.read_bytes())

def ler_jsonl(path):
    return [json.loads(linha) for linha in path.read_text(encoding='utf-8').splitlines() if linha.strip()]

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def tabela(colunas, linhas):
    cabecalho = ''.join('<th>' + html.escape(str(c)) + '</th>' for c in colunas)
    corpo = ''.join('<tr>' + ''.join('<td>' + html.escape(str(v)) + '</td>' for v in linha) + '</tr>' for linha in linhas)
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + cabecalho + '</tr></thead><tbody>' + corpo + '</tbody></table></div>'))

def percentual(taxa):
    return f'{100 * taxa:.2f}%'.replace('.', ',')

def salvar_csv(nome, colunas, linhas):
    with (SAIDA / 'tabelas' / nome).open('w', encoding='utf-8-sig', newline='') as arquivo:
        gravador = csv.writer(arquivo)
        gravador.writerow(colunas)
        gravador.writerows(linhas)

AMBIENTE = {p: importlib.metadata.version(p) for p in ['jiwer', 'Pillow', 'numpy', 'matplotlib', 'nbformat', 'nbclient']}
tabela(['Componente', 'Versão'], [('Python', platform.python_version()), *AMBIENTE.items()])

## 2. Integridade e preservação

O auditor existente do experimento confere hashes, pixels, ordem, 24 pares por modelo,
ressalvas e métricas. Ele também verifica o número de operações por uma implementação
independente de Levenshtein. Não precisa dos pesos de reconhecimento.

In [ ]:
INVENTARIO = ler_json(PACOTE / 'inventario.json')
HASHES_ANTES = {nome: digest(PACOTE / nome) for nome in INVENTARIO}
assert HASHES_ANTES == INVENTARIO
auditoria = runpy.run_path(str(PACOTE / 'verificar.py'))['verificar']()
assert auditoria['resultado'] == 'aprovado'

WORKSPACE = RAIZ / 'data/workspace_traslado_269r'
WORKSPACE_ANTES = ({p.relative_to(RAIZ).as_posix(): digest(p) for p in WORKSPACE.rglob('*') if p.is_file()}
                   if WORKSPACE.exists() else None)
tabela(['Verificação', 'Resultado'], [
    ('Arquivos do experimento conferidos', auditoria['arquivos_conferidos']),
    ('Linhas de cada modelo', auditoria['linhas']),
    ('Blocos completos', auditoria['blocos']),
    ('Ressalvas preservadas', auditoria['ressalvas_preservadas']),
    ('Distâncias conferidas independentemente', auditoria['distancias_conferidas_por_algoritmo_independente']),
])

## 3. Corpus e política textual

O conjunto contém 24 linhas consecutivas do corpo do 269r, sem cabeçalho ou verso.
A referência combina conferência humana e apoio da edição publicada nas dúvidas.
Mantêm-se `Eram`, `Com`, `Como`, o espaçamento `que mais` e a abreviatura `Magᵈᵉ`.
`Magestade` permanece desenvolvimento editorial, sem substituir o alvo abreviado.

Na versão congelada de 20/10, há 16 linhas com estado validado e oito com ressalvas históricas. Todas entram na métrica.
Os alvos assistidos não são promovidos a referência humana independente. As
ambiguidades de palavras, pontuação e abreviaturas são parte da interpretação do resultado.

In [ ]:
ALVOS = ler_jsonl(PACOTE / 'alvos_assistidos.jsonl')
ORIGEM = ler_jsonl(PACOTE / 'catalogo_origem.jsonl')
RECORTES = ler_jsonl(PACOTE / 'recortes_reconhecimento.jsonl')
PROTOCOLO = ler_json(PACOTE / 'protocolo.json')
SALVO = ler_json(PACOTE / 'comparacao.json')
PREDICOES = {modelo: ler_jsonl(PACOTE / modelo / 'predictions.jsonl') for modelo in ['ppocrv5', 'ppocrv6']}
MODELOS = {'ppocrv5': 'PP-OCRv5 latino mobile', 'ppocrv6': 'PP-OCRv6 medium'}
RESSALVAS = [a for a in ALVOS if a['estado_visual_original'] != 'validated']
assert [a['linha'] for a in RESSALVAS] == [11, 15, 16, 17, 19, 20, 23, 24]
assert [a['texto_alvo'] for a in ALVOS] == [o['text_diplomatic'] for o in ORIGEM]
assert len(ALVOS) == 24 and not PROTOCOLO['teste_reservado']
assert not PROTOCOLO['referencia_humana_independente']
assert all(r['text_diplomatic'] is None and r['reference_kind'] == 'unknown' for r in RECORTES)

linhas_ressalvas = [(a['linha'], a['texto_alvo'], '; '.join(d['trecho'] + ': ' + d['duvida'] for d in a['duvidas_preservadas'])) for a in RESSALVAS]
tabela(['Linha', 'Alvo adotado', 'Ressalva preservada'], linhas_ressalvas)
salvar_csv('ressalvas.csv', ['linha', 'alvo_adotado', 'ressalva_preservada'], linhas_ressalvas)

## 3.1. Reconferência posterior das oito ressalvas

Em 21/10/2026, **Rodolfo Vinícius Cima Takemoto** reconferiu os trechos
registrados. A consolidação foi publicada depois do experimento. A tabela
separa o alvo e a ressalva originais das confirmações pontuais, das escolhas
apoiadas na edição e das dúvidas atuais. Os vínculos apontam para evidências
no commit publicado; o CSV conserva caminhos e hashes.

As linhas **15, 16 e 20** receberam confirmações pontuais; a **17** combina
leitura visual e escolha editorial de `ministros`, conservando `menistros` no
histórico. Continuam abertas a natureza do sinal em **Requererão** (11), a
inicial **o/e** de **orgente** (19), os **dois s** de **Justissa** (23) e a
base **Mag provável** e a **pontuação final** (24). O `de` elevado foi reconfirmado.

Isso não é uma nova contagem de linhas validadas nem uma leitura independente
da edição. Nenhum texto desta tabela substitui os alvos congelados. A
abreviatura `Magᵈᵉ` não é substituída por seu desenvolvimento editorial `Magestade`.


In [ ]:
RECONFERENCIA = RAIZ / 'docs/revisao_humana/traslado_269r/reconferencia_ressalvas_20261021'
ENTREGA_ANTERIOR = RAIZ / 'docs/entregas/pi_iv_pln_20261020'
HASHES_ENTREGA_ANTERIOR = {p.relative_to(ENTREGA_ANTERIOR).as_posix(): digest(p)
                           for p in ENTREGA_ANTERIOR.rglob('*') if p.is_file() and '__pycache__' not in p.parts}
INVENTARIO_RECONFERENCIA = ler_json(RECONFERENCIA / 'inventario.json')
HASH_INVENTARIO_RECONFERENCIA = digest(RECONFERENCIA / 'inventario.json')
assert {p.name for p in RECONFERENCIA.iterdir() if p.is_file()} == set(INVENTARIO_RECONFERENCIA) | {'inventario.json'}
assert all(digest(RECONFERENCIA / nome) == h for nome, h in INVENTARIO_RECONFERENCIA.items())
CONSOLIDACAO = ler_json(RECONFERENCIA / 'consolidacao.json')
assert CONSOLIDACAO['natureza'] == 'diagnóstico de desenvolvimento'
assert not CONSOLIDACAO['referencia_humana_independente_da_edicao']
assert not CONSOLIDACAO['duvidas_foram_zeradas'] and not CONSOLIDACAO['gravado_no_editor']
assert CONSOLIDACAO['novo_total_de_linhas_validadas_no_catalogo'] is None
assert CONSOLIDACAO['referencia_congelada']['ressalvas'] == [a['linha'] for a in RESSALVAS]
assert CONSOLIDACAO['novas_confirmacoes_pontuais_de_ressalvas'] == [15, 16, 20]
assert CONSOLIDACAO['linhas_com_duvidas_visuais_expressamente_mantidas'] == [11, 19, 23, 24]
assert len(CONSOLIDACAO['historico']['eventos_humanos_ordenados']) == 17
for evento in CONSOLIDACAO['historico']['eventos_humanos_ordenados']:
    assert digest(RECONFERENCIA / evento['arquivo']) == evento['sha256']

DECISOES = CONSOLIDACAO['resultados_por_linha']
assert [d['linha'] for d in DECISOES] == [a['linha'] for a in RESSALVAS]
cabecalho = ['linha', 'alvo_congelado_20261020', 'ressalva_original_20261020',
             'decisao_reconferencia_20261021', 'confirmacoes_visuais_pontuais',
             'apoio_editorial', 'duvidas_visuais_mantidas', 'notas_de_escopo_e_historico',
             'revisor', 'evidencias_caminhos', 'evidencias_sha256']
exportacao = []
permalink = 'https://github.com/aj1no/paleographia-htr/blob/27b547d0203b566ca5cf17c1378f64ce46ae5b1a/'
display(Markdown('**Revisor:** ' + CONSOLIDACAO['revisor'] + '. Catálogo e alvos originais preservados.'))
for d, a, original in zip(DECISOES, RESSALVAS, linhas_ressalvas):
    assert not d['texto_congelado_alterado'] and d['texto_adotado_na_referencia_congelada'] == a['texto_alvo']
    evidencias = [(RECONFERENCIA / nome).relative_to(RAIZ).as_posix() for nome in d['decisoes']]
    hashes = [digest(RAIZ / nome) for nome in evidencias]
    notas = [d[k] for k in ['escopo', 'observacao_visual_do_revisor', 'confirmacao_anterior_preservada', 'outra_observacao'] if k in d]
    notas += d.get('historico_preservado', []) + d.get('nao_afirmado', [])
    exportacao.append((d['linha'], a['texto_alvo'], original[2], d['trecho_adotado'],
                       '; '.join(d['confirmacoes_visuais_novas']), '; '.join(d['apoio_editorial']),
                       '; '.join(d['duvidas_visuais_preservadas']), '; '.join(notas),
                       CONSOLIDACAO['revisor'], '; '.join(evidencias), '; '.join(hashes)))
    display(Markdown('### Linha ' + str(d['linha'])))
    tabela(['Campo', 'Registro'], [
        ('Alvo congelado de 20/10', a['texto_alvo']), ('Ressalva original', original[2]),
        ('Decisão de 21/10', d['trecho_adotado']),
        ('Confirmações visuais pontuais', '; '.join(d['confirmacoes_visuais_novas']) or 'Nenhuma nova confirmação isolada'),
        ('Apoio editorial', '; '.join(d['apoio_editorial']) or 'Nenhuma nova adoção editorial'),
        ('Dúvidas visuais mantidas', '; '.join(d['duvidas_visuais_preservadas']) or 'Nenhuma dúvida adicional listada para este trecho; sem promover a linha inteira'),
        ('Notas de escopo e histórico', '; '.join(notas)),
    ])
    display(Markdown('Evidências: ' + ', '.join('[' + Path(nome).name + '](' + permalink + nome + ')' for nome in evidencias)))
salvar_csv('reconferencia_ressalvas.csv', cabecalho, exportacao)


## 4. Recalcular CER e WER

Usa-se Unicode NFC e colapso de espaços em ambos os lados, preservando caixa,
acentos e grafia histórica. Para cada bloco completo, as linhas são concatenadas
em ordem com um espaço. As contagens dos cinco blocos são somadas antes de dividir:

`CER = (substituições + deleções + inserções) / caracteres de referência`

`WER = (substituições + deleções + inserções) / palavras de referência`

Não se calcula a média das taxas por linha. Falha de reconhecimento equivale a
saída vazia. Não se descartam linhas difíceis nem se limita a taxa a 100%.

In [ ]:
GRUPOS = defaultdict(list)
for alvo in ALVOS:
    GRUPOS[alvo['region_id']].append(alvo)
assert len(GRUPOS) == 5
RECALCULADO, POR_BLOCO = {}, []
for modelo, predicoes in PREDICOES.items():
    assert [p['line_id'] for p in predicoes] == [a['line_id'] for a in ALVOS]
    assert len(predicoes) == len({p['line_id'] for p in predicoes}) == 24
    indice = {p['line_id']: p for p in predicoes}
    contagens = []
    for regiao, alvos_bloco in GRUPOS.items():
        texto_ref = ' '.join(a['texto_alvo'] for a in alvos_bloco)
        texto_pred = ' '.join(indice[a['line_id']]['text_raw'] if indice[a['line_id']]['status'] == 'ok' else '' for a in alvos_bloco)
        metricas = counts(texto_ref, texto_pred)
        contagens.append(metricas)
        POR_BLOCO.append({'modelo': modelo, 'bloco': f"{alvos_bloco[0]['linha']}–{alvos_bloco[-1]['linha']}",
                          'metricas': metricas})
    RECALCULADO[modelo] = aggregate(contagens)
    assert RECALCULADO[modelo] == SALVO['modelos'][modelo]['metrics']

linhas_resumo, linhas_numericas = [], []
for modelo, valores in RECALCULADO.items():
    c, p = valores['characters'], valores['words']
    falhas = sum(x['status'] != 'ok' for x in PREDICOES[modelo])
    linhas_resumo.append((MODELOS[modelo], percentual(c['rate']), f"{c['errors']}/{c['reference_length']}",
                          percentual(p['rate']), f"{p['errors']}/{p['reference_length']}", falhas))
    linhas_numericas.append((modelo, c['rate'], c['errors'], c['reference_length'], p['rate'], p['errors'], p['reference_length'], falhas))
tabela(['Reconhecedor', 'CER', 'Erros/caracteres', 'WER', 'Erros/palavras', 'Falhas'], linhas_resumo)
salvar_csv('resumo_metricas.csv', ['modelo', 'CER', 'erros_caracteres', 'caracteres_referencia', 'WER', 'erros_palavras', 'palavras_referencia', 'falhas'], linhas_numericas)
assert all(m['characters']['reference_length'] == 1112 and m['words']['reference_length'] == 203 for m in RECALCULADO.values())

## 5. Figura principal

As barras apresentam taxas de erro, portanto menores valores indicam menos
divergências em relação ao alvo assistido. Não há barras de confiança: este
diagnóstico de um documento não estima incerteza de generalização.

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'svg.hashsalt': 'paleographia-269r24',
                     'axes.spines.top': False, 'axes.spines.right': False})
cores = {'ppocrv5': '#315b7d', 'ppocrv6': '#18816e'}
fig, ax = plt.subplots(figsize=(8.4, 4.4), layout='constrained')
for deslocamento, modelo in [(-0.18, 'ppocrv5'), (0.18, 'ppocrv6')]:
    taxas = [100 * RECALCULADO[modelo][m]['rate'] for m in ['characters', 'words']]
    barras = ax.bar([deslocamento, 1+deslocamento], taxas, width=.34, label=MODELOS[modelo], color=cores[modelo])
    ax.bar_label(barras, labels=[f'{v:.2f}%'.replace('.', ',') for v in taxas], padding=4, fontsize=10)
ax.set(xticks=[0, 1], xticklabels=['CER', 'WER'], ylim=(0, 108), ylabel='Taxa de erro (%)',
       title='24 linhas • referência assistida • oito ressalvas preservadas')
fig.suptitle('Diagnóstico de desenvolvimento — fólio 269r', fontsize=12)
ax.yaxis.set_major_locator(MultipleLocator(20))
ax.grid(axis='y', alpha=.18)
ax.set_axisbelow(True)
ax.legend(loc='upper left', frameon=False)
fig.savefig(SAIDA / 'figuras/cer_wer.png', dpi=180, metadata={'Software': 'PaleographIA / Matplotlib'})
fig.savefig(SAIDA / 'figuras/cer_wer.svg', metadata={'Date': None})
plt.show()

## 6. Contagens de operações e blocos completos

A comparação principal contém 650 operações em caracteres para o PP-OCRv5 e
168 para o PP-OCRv6; estes totais serão novamente obtidos dos dados, sem transcrição
manual de resultados. As taxas por bloco são descritivas, sem média simples.

In [ ]:
operacoes = []
for modelo, metricas in RECALCULADO.items():
    for unidade, nome in [('characters', 'caracteres'), ('words', 'palavras')]:
        m = metricas[unidade]
        operacoes.append((MODELOS[modelo], nome, m['substitutions'], m['deletions'], m['insertions'], m['errors'], m['reference_length']))
tabela(['Modelo', 'Unidade', 'Substituições', 'Deleções', 'Inserções', 'Erros', 'Denominador'], operacoes)
salvar_csv('operacoes.csv', ['modelo', 'unidade', 'substituicoes', 'delecoes', 'insercoes', 'erros', 'denominador'], operacoes)
blocos_tabela = [(MODELOS[r['modelo']], r['bloco'], percentual(r['metricas']['characters']['rate']), percentual(r['metricas']['words']['rate'])) for r in POR_BLOCO]
tabela(['Modelo', 'Linhas', 'CER', 'WER'], blocos_tabela)
salvar_csv('metricas_blocos.csv', ['modelo', 'linhas', 'CER', 'WER'], [(r['modelo'], r['bloco'], r['metricas']['characters']['rate'], r['metricas']['words']['rate']) for r in POR_BLOCO])

fig, ax = plt.subplots(figsize=(8.4, 4.2), layout='constrained')
for modelo in MODELOS:
    dados = [r for r in POR_BLOCO if r['modelo'] == modelo]
    ax.plot(range(len(dados)), [100*r['metricas']['characters']['rate'] for r in dados], marker='o', linewidth=2,
            color=cores[modelo], label=MODELOS[modelo])
ax.set(xticks=range(5), xticklabels=[r['bloco'] for r in POR_BLOCO if r['modelo'] == 'ppocrv5'],
       ylim=(0, 100), xlabel='Linhas do bloco completo', ylabel='CER (%)', title='Referência assistida • oito ressalvas preservadas')
fig.suptitle('Diagnóstico de desenvolvimento: CER por bloco', fontsize=12)
ax.grid(axis='y', alpha=.18)
ax.legend(loc='upper left', frameon=False)
fig.savefig(SAIDA / 'figuras/cer_blocos.png', dpi=180, metadata={'Software': 'PaleographIA / Matplotlib'})
fig.savefig(SAIDA / 'figuras/cer_blocos.svg', metadata={'Date': None})
plt.show()

## 7. Diagnóstico por linha e exemplos

As 24 linhas continuam presentes. Uma taxa por linha pode superar 100% por causa
de inserções; ela é diagnóstica e não substitui a contagem por blocos. A tabela
abaixo destaca exemplos; o CSV exportado contém todas as linhas dos dois modelos.

Os exemplos permitem observar perda de conteúdo no PP-OCRv5 e, no PP-OCRv6,
junções/separações de palavras, trocas de letras, caixa e sinais diacríticos.
As divergências de palavras que já tinham ressalvas não resolvem a paleografia.

In [ ]:
linhas_diagnostico, exportar_linhas = [], []
for alvo in ALVOS:
    por_modelo = {}
    for modelo in MODELOS:
        pred = next(p for p in PREDICOES[modelo] if p['line_id'] == alvo['line_id'])
        texto = pred['text_raw'] if pred['status'] == 'ok' else ''
        metricas = counts(alvo['texto_alvo'], texto)
        por_modelo[modelo] = texto
        exportar_linhas.append((modelo, alvo['linha'], alvo['texto_alvo'], texto,
                               alvo['estado_visual_original'], metricas['characters']['rate'], metricas['words']['rate']))
    if alvo['linha'] in [1, 6, 14, 22, 23, 24]:
        linhas_diagnostico.append((alvo['linha'], alvo['texto_alvo'], por_modelo['ppocrv5'], por_modelo['ppocrv6'],
                                  'Sim' if alvo['estado_visual_original'] != 'validated' else 'Não'))
tabela(['Linha', 'Alvo assistido', 'PP-OCRv5 bruto', 'PP-OCRv6 bruto', 'Ressalva'], linhas_diagnostico)
salvar_csv('24_linhas.csv', ['modelo', 'linha', 'alvo', 'predicao_bruta', 'estado_visual_original', 'CER_linha', 'WER_linha'], exportar_linhas)
assert len(exportar_linhas) == 48

## 8. Alinhamentos frequentes

Esta contagem usa os alinhamentos **por linha** arquivados no experimento.
Serve para localizar exemplos; não deve ser somada para substituir os totais
principais por bloco. O símbolo `␠` apenas torna espaços visíveis na tabela.

In [ ]:
erros = ler_jsonl(PACOTE / 'erros_caracteres.jsonl')
rotulos = {'delete': 'Deleção', 'insert': 'Inserção', 'substitute': 'Substituição'}
for modelo in MODELOS:
    frequencias = Counter((e['operacao'], e['referencia'], e['predicao']) for e in erros if e['modelo'] == modelo)
    display(Markdown('**' + MODELOS[modelo] + '**'))
    tabela(['Operação', 'Referência', 'Predição', 'Ocorrências por linha'],
           [(rotulos[tipo], ref.replace(' ', '␠') or '∅', pred.replace(' ', '␠') or '∅', total)
            for (tipo, ref, pred), total in frequencias.most_common(6)])

## 9. Interpretação e limites

- O PP-OCRv6 apresenta menos divergências neste único documento, ainda com WER elevado.
- Um erro de espaço, caixa ou acento pode afetar a correspondência de palavras inteiras.
- Há oito ressalvas; a referência é assistida pela edição e não independente dela.
- O fólio já foi usado em desenvolvimento e consultas lexicais. Não é teste reservado.
- As 24 linhas de um mesmo documento não são 24 amostras documentais independentes.
- Variantes, corpora de pré-treino e implementações diferem. Não se isola um efeito causal
  da versão e não foi auditada eventual sobreposição com o pré-treino externo.
- Na comparação de 20/10 não houve treino, ajuste fino, pós-correção, nova segmentação ou estudo de esforço humano.
- A meta inicial de 30–50 linhas não é garantia de suficiência; a expansão de 23/10 não torna o 269r reservado.

O artigo de 23/10 está diagramado no modelo encontrado; o enquadramento docente
e a revisão sistemática de PI IV continuam pendentes.

A reconferência de 21/10 é posterior ao experimento e muda a interpretação
documentada de alguns trechos, sem alterar a referência avaliada. Não se deve
atribuir a manutenção das mesmas métricas a uma melhora dos reconhecedores.


## 10. Preservação final e saídas derivadas

Conferir que os arquivos históricos continuam idênticos e que o workspace, se
presente neste checkout, não foi alterado. Os CSVs e as figuras vão apenas para `output/`.

In [ ]:
assert {nome: digest(PACOTE / nome) for nome in INVENTARIO} == HASHES_ANTES
if WORKSPACE_ANTES is not None:
    workspace_depois = {p.relative_to(RAIZ).as_posix(): digest(p) for p in WORKSPACE.rglob('*') if p.is_file()}
    assert workspace_depois == WORKSPACE_ANTES
assert {p.relative_to(ENTREGA_ANTERIOR).as_posix(): digest(p)
        for p in ENTREGA_ANTERIOR.rglob('*') if p.is_file() and '__pycache__' not in p.parts} == HASHES_ENTREGA_ANTERIOR
assert all(digest(RECONFERENCIA / nome) == h for nome, h in INVENTARIO_RECONFERENCIA.items())
assert digest(RECONFERENCIA / 'inventario.json') == HASH_INVENTARIO_RECONFERENCIA
resultado = {
    'tipo': 'diagnostico_de_desenvolvimento', 'linhas': 24, 'ressalvas': 8,
    'regioes': len(GRUPOS), 'metricas': RECALCULADO,
    'arquivos_experimento_inalterados': len(INVENTARIO),
    'workspace_conferido': WORKSPACE_ANTES is not None,
    'workspace_arquivos_inalterados': len(WORKSPACE_ANTES) if WORKSPACE_ANTES is not None else None,
    'novas_inferencias': 0, 'treino': False, 'teste_reservado': False,
    'reconferencia': {'data': '2026-10-21', 'linhas': [d['linha'] for d in DECISOES],
                       'registros_humanos': 17, 'catalogo_promovido': False, 'alvos_alterados': False,
                       'leitura_independente_da_edicao': False,
                       'inventario_sha256': HASH_INVENTARIO_RECONFERENCIA,
                       'pacote_20261020_preservado': True},
    'ambiente': AMBIENTE, 'python': platform.python_version(),
}
(SAIDA / 'resultado_notebook.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
display(Markdown('**Auditoria concluída.** Pacote histórico preservado; nenhuma nova inferência ou alteração do editor.'))
tabela(['Artefato derivado', 'Pasta relativa à raiz'], [('Figuras PNG/SVG', (SAIDA / 'figuras').relative_to(RAIZ)),
                                                      ('Tabelas CSV', (SAIDA / 'tabelas').relative_to(RAIZ)),
                                                      ('Recibo numérico', (SAIDA / 'resultado_notebook.json').relative_to(RAIZ))])

## Fontes e articulação com a entrega

- [Artigo preparado para PI IV e PLN](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/entregas/pi_iv_pln_20261023_entrega/README.md).
- [Fontes e conferência de citações](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/entregas/pi_iv_pln_20261023_entrega/citacoes.json).
- [Fichamentos completos T1/T4 por IA](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/bibliografia/leitura_t1_t4_20261023/README.md).
- [Notebook histórico de 22/10](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/entregas/pi_iv_pln_20261022/README.md), preservado.
- [Experimento e reprodução opcional de inferências](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/experimentos/traslado_269r_reconhecedores_24_20261020/README.md).
- [Expansão e ensaio técnico posteriores](https://github.com/aj1no/paleographia-htr/blob/c8a3e9a7f1cc55b66d630c510eff17ff95dfdece/docs/dataset_expansion_20261023/README.md).

Recalcular métricas sobre saídas arquivadas não é nova inferência. A discussão
abaixo deve considerar o diagnóstico de desenvolvimento e as oito ressalvas.


# Participação individual

As tabelas e imagens desta seção são geradas automaticamente como material de
trabalho. Elas **não são respostas de Caio, Guilherme, Tiago ou Rodolfo**.
Cada integrante preenche apenas o seu relato. Use também este espaço para
documentar uma contribuição concreta da discussão já ocorrida, sem precisar
alegar programação ou autoria de uma execução feita por outra pessoa.

**Formulário:** `preenchido_por` = quem está escrevendo; `data_atividade` = data
real (AAAA-MM-DD); `relato` = contribuição e interpretação próprias;
`evidencias` = caminhos de arquivos no repositório, URLs ou `relato: descrição
da discussão`; `uso_de_ia` = ferramenta e função, ou declaração de não utilização.
Se alguém registrar o relato de um colega, identifique quem preencheu; o
colega ainda precisa conferir. Os registros são autodeclarações, sem autenticação
ou aprovação automática e sem validação de linhas do manuscrito.

In [ ]:
EQUIPE_DIR = RAIZ / 'docs/entregas/pi_iv_pln_20261023_equipe'
PARTICIPACAO = runpy.run_path(str(EQUIPE_DIR / 'participacao.py'))
registrar_participacao = PARTICIPACAO['registrar']
listar_participacoes = PARTICIPACAO['registros']
modelo_formulario = PARTICIPACAO['formulario_vazio']
print('Formulários preparados; nenhuma participação individual foi atribuída pela execução.')

## Caio — bibliografia e fundamentação

**Previsto no primeiro artigo:** revisão bibliográfica de HTR/segmentação.
**Atividade atual:** conferir as relações entre T1/T4 e o artigo do diagnóstico.

Abra as fontes e os fichamentos nas referências abaixo. Explique, com página,
uma afirmação que T1 sustenta e um limite de usar seus casos como comparação.
Em T4, identifique uma condição na qual o ajuste não melhora o CER e explique
por que NFD/macro não equivale a NFC/micro local. Aponte uma sugestão de revisão
ou uma concordância justificada com a fundamentação. Se sua contribuição já
ocorreu na conversa do grupo, descreva a proposta ou a dúvida que você trouxe.

Preencha os campos do formulário e execute a célula. As respostas não são pré-escritas.

In [ ]:
# @title Caio — preencher e executar { display-mode: "form" }
FONTES_ARTIGO = ler_json(RAIZ / 'docs/entregas/pi_iv_pln_20261023_entrega/citacoes.json')
tabela(['Fonte', 'Uso previsto no artigo', 'Escopo da consulta registrada'],
       [(r['key'], '; '.join(r['uses']), r['scope']) for r in FONTES_ARTIGO['references']
        if r['key'] in ['muehlberger2019', 'kiessling2026a', 'kiessling2026b']])
preenchido_por_caio = "" # @param {type:"string", placeholder:"Nome de quem preenche"}
data_atividade_caio = "" # @param {type:"string", placeholder:"AAAA-MM-DD, data real"}
relato_caio = "" # @param {type:"string", placeholder:"Descreva sua contribuição concreta"}
evidencias_caio = "" # @param {type:"string", placeholder:"Caminho, URL ou relato: descrição; separe itens com ponto e vírgula"}
uso_de_ia_caio = "" # @param {type:"string", placeholder:"Ferramenta e função, ou declaração de não utilização"}
resposta_caio = {
    "preenchido_por": preenchido_por_caio,
    "data_atividade": data_atividade_caio,
    "relato": relato_caio,
    "evidencias": [e.strip() for e in evidencias_caio.split(";") if e.strip()],
    "uso_de_ia": uso_de_ia_caio,
}
resultado_caio = registrar_participacao(RAIZ, "caio", resposta_caio)
print(resultado_caio)

## Guilherme — modelos e reprodução

**Previsto no primeiro artigo:** avaliação de modelos/dependências e pipeline.
**Atividade atual:** verificar as configurações dos dois reconhecedores e a
reprodução dos cálculos, usando os registros já disponíveis.

Compare os identificadores, hashes declarados, dispositivo e precisão das
proveniências. Relacione essa comparação ao resumo CER/WER recalculado.
Explique duas diferenças de configuração que impedem atribuir o resultado só
à versão da arquitetura. Distinga reexecução do cálculo de nova inferência e
de ajuste fino. Registre uma checagem que você fez ou uma questão técnica que
levantou na discussão. O notebook lê a proveniência; não carrega pesos para
comprovar novamente o hash do arquivo do modelo.

In [ ]:
# @title Guilherme — preencher e executar { display-mode: "form" }
PROV5 = ler_json(PACOTE / 'ppocrv5/provenance.json')
PROV6 = ler_json(PACOTE / 'ppocrv6/provenance.json')
PESO5 = PROV5['identidade']['loaded_model']
CFG6 = PROV6['config']['recognition']
tabela(['Configuração', 'Identidade declarada', 'SHA-256 declarado'], [
    ('PP-OCRv5 / ONNX / CPU / float32', PESO5['id'], PESO5['sha256']),
    ('PP-OCRv6 / Kraken / ' + CFG6['device'] + ' / ' + CFG6['precision'], CFG6['model'], CFG6['sha256']),
])
preenchido_por_guilherme = "" # @param {type:"string", placeholder:"Nome de quem preenche"}
data_atividade_guilherme = "" # @param {type:"string", placeholder:"AAAA-MM-DD, data real"}
relato_guilherme = "" # @param {type:"string", placeholder:"Descreva sua contribuição concreta"}
evidencias_guilherme = "" # @param {type:"string", placeholder:"Caminho, URL ou relato: descrição; separe itens com ponto e vírgula"}
uso_de_ia_guilherme = "" # @param {type:"string", placeholder:"Ferramenta e função, ou declaração de não utilização"}
resposta_guilherme = {
    "preenchido_por": preenchido_por_guilherme,
    "data_atividade": data_atividade_guilherme,
    "relato": relato_guilherme,
    "evidencias": [e.strip() for e in evidencias_guilherme.split(";") if e.strip()],
    "uso_de_ia": uso_de_ia_guilherme,
}
resultado_guilherme = registrar_participacao(RAIZ, "guilherme", resposta_guilherme)
print(resultado_guilherme)

## Tiago — imagens, associações e erros

**Previsto no primeiro artigo:** inventário e associações imagem–texto com
Rodolfo, além de colaboração no pipeline tradicional com Guilherme.
**Atividade atual:** inspecionar o vínculo entre os recortes e o diagnóstico.

Confira as linhas 1, 17 e 24: imagem, ID, coordenadas e correspondência com a
linha física. Compare os resultados brutos e descreva um problema de recorte,
espaçamento, letra ou abreviatura que você observou. Diferencie erro do modelo
de uma dúvida já existente no alvo. Informe no relato se consultou a edição,
o alvo ou as saídas; esta atividade não presume leitura independente. Ela não
altera o catálogo nem promove uma ressalva a validação. Na próxima rodada,
o mesmo método pode apoiar o inventário dos novos candidatos do corpus.

In [ ]:
# @title Tiago — preencher e executar { display-mode: "form" }
from IPython.display import Image

for numero in [1, 17, 24]:
    alvo = next(a for a in ALVOS if a['linha'] == numero)
    recorte = next(r for r in RECORTES if r['line_id'] == alvo['line_id'])
    display(Markdown('### Inspeção proposta — linha ' + str(numero)))
    display(Image(filename=str(RAIZ / recorte['crop_image']), width=800))
    tabela(['Campo', 'Informação arquivada'], [
        ('ID', recorte['line_id']), ('Coordenadas', recorte['bbox_xyxy']),
        ('Imagem de origem', recorte['source_image']),
        ('Alvo assistido', alvo['texto_alvo']), ('Estado histórico', alvo['estado_visual_original']),
        *[(MODELOS[m], next(p['text_raw'] for p in PREDICOES[m] if p['line_id'] == alvo['line_id'])) for m in MODELOS],
    ])
preenchido_por_tiago = "" # @param {type:"string", placeholder:"Nome de quem preenche"}
data_atividade_tiago = "" # @param {type:"string", placeholder:"AAAA-MM-DD, data real"}
relato_tiago = "" # @param {type:"string", placeholder:"Descreva sua contribuição concreta"}
evidencias_tiago = "" # @param {type:"string", placeholder:"Caminho, URL ou relato: descrição; separe itens com ponto e vírgula"}
uso_de_ia_tiago = "" # @param {type:"string", placeholder:"Ferramenta e função, ou declaração de não utilização"}
resposta_tiago = {
    "preenchido_por": preenchido_por_tiago,
    "data_atividade": data_atividade_tiago,
    "relato": relato_tiago,
    "evidencias": [e.strip() for e in evidencias_tiago.split(";") if e.strip()],
    "uso_de_ia": uso_de_ia_tiago,
}
resultado_tiago = registrar_participacao(RAIZ, "tiago", resposta_tiago)
print(resultado_tiago)

## Rodolfo — coordenação e consolidação

**Previsto no primeiro artigo:** dados, associações e pipeline, além das
entregas da equipe. **Evidência histórica disponível:** os 17 registros humanos
da reconferência de 21/10, mostrados na parte comum. Não precisam ser reatribuídos.

Nesta rodada, reúna as respostas dos colegas e confira se o relato distingue
planejamento, discussão e execução. Registre também sua própria implementação,
revisão ou coordenação efetivamente realizada, indicando arquivos, commits ou
outros registros. As contribuições dos colegas devem ser conferidas por eles.
Não preencha o formulário alheio como se fosse confirmação do participante.

In [ ]:
# @title Rodolfo — preencher e executar { display-mode: "form" }
preenchido_por_rodolfo = "" # @param {type:"string", placeholder:"Nome de quem preenche"}
data_atividade_rodolfo = "" # @param {type:"string", placeholder:"AAAA-MM-DD, data real"}
relato_rodolfo = "" # @param {type:"string", placeholder:"Descreva sua contribuição concreta"}
evidencias_rodolfo = "" # @param {type:"string", placeholder:"Caminho, URL ou relato: descrição; separe itens com ponto e vírgula"}
uso_de_ia_rodolfo = "" # @param {type:"string", placeholder:"Ferramenta e função, ou declaração de não utilização"}
resposta_rodolfo = {
    "preenchido_por": preenchido_por_rodolfo,
    "data_atividade": data_atividade_rodolfo,
    "relato": relato_rodolfo,
    "evidencias": [e.strip() for e in evidencias_rodolfo.split(";") if e.strip()],
    "uso_de_ia": uso_de_ia_rodolfo,
}
resultado_rodolfo = registrar_participacao(RAIZ, "rodolfo", resposta_rodolfo)
print(resultado_rodolfo)

## Consolidação da rodada e preservação

A tabela abaixo só lista arquivos realmente encontrados na pasta de registros.
Ausência de registro individual não apaga uma discussão coletiva; significa
que a contribuição ainda não foi individualizada aqui. A existência de JSON
também não autentica identidade nem prova revisão independente do manuscrito.

Depois da rodada, Caio e Guilherme podem conferir os relatos um do outro;
Tiago e Rodolfo podem fazer o mesmo. São revisões propostas, ainda não feitas.
A coordenação poderá usar os relatos conferidos para redigir a contribuição
dos autores em uma nova versão do artigo. O artigo já publicado não é alterado
automaticamente por estes formulários.

In [ ]:
REGISTROS_EQUIPE = listar_participacoes(RAIZ)
tabela(['Participante previsto', 'Registros individuais disponíveis', 'Situação'], [
    (nome, sum(r['participante_id'] == pid for r in REGISTROS_EQUIPE),
     'Relato(s) disponível(is), por conferir' if any(r['participante_id'] == pid for r in REGISTROS_EQUIPE)
     else 'Contribuição individual a preencher')
    for pid, nome in PARTICIPACAO['PARTICIPANTES'].items()
])
assert {nome: digest(PACOTE / nome) for nome in INVENTARIO} == HASHES_ANTES
assert digest(RECONFERENCIA / 'inventario.json') == HASH_INVENTARIO_RECONFERENCIA
if WORKSPACE_ANTES is not None:
    assert {p.relative_to(RAIZ).as_posix(): digest(p) for p in WORKSPACE.rglob('*') if p.is_file()} == WORKSPACE_ANTES
RESUMO_EQUIPE = {
    'tipo': 'roteiro_colaborativo_com_registros_autodeclarados',
    'formularios_desta_execucao': {pid: resultado for pid, resultado in [
        ('caio', resultado_caio), ('guilherme', resultado_guilherme),
        ('tiago', resultado_tiago), ('rodolfo', resultado_rodolfo)]},
    'registros_individuais_encontrados': len(REGISTROS_EQUIPE),
    'identidade_autenticada': False, 'participacao_individual_inferida_do_cronograma': False,
    'novas_inferencias': 0, 'catalogo_alterado': False,
}
(SAIDA.parent / 'resumo_participacao.json').write_text(json.dumps(RESUMO_EQUIPE, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Diagnóstico preservado. Relatos individuais são registros separados, sujeitos à conferência.')

## Comentários sobre uso de IA

Ferramenta: Codex, da OpenAI. Utilização: organização deste roteiro a partir
do planejamento inicial e do relato do solicitante, preparação das células,
reprodução dos cálculos e verificações. Os formulários individuais da execução
de demonstração estão vazios. Nenhuma resposta dos colegas foi escrita ou
confirmada pela IA. Cada participante deve registrar seu próprio uso de IA.

## Baixar minha contribuição

Depois de executar seu formulário, escolha seu nome, marque a caixa e execute
a célula abaixo. O ZIP reúne **somente os registros do integrante selecionado**
encontrados nesta sessão, incluindo versões sucessivas. Conferir e incorporar
esses registros à documentação continua sendo tarefa da equipe.

Se corrigir um campo, execute seu formulário novamente e baixe outro ZIP.
Você pode rodar novamente a consolidação acima para atualizar a tabela.


In [ ]:
# @title Baixar minha contribuição { display-mode: "form" }
"""Pacote de contribuições de um participante, sem autenticar ou publicar relatos."""
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED
import json


def criar_pacote_contribuicoes(raiz, participante):
    if participante not in {'caio', 'guilherme', 'tiago', 'rodolfo'}:
        raise ValueError('Selecione um integrante da equipe.')
    raiz = Path(raiz).resolve()
    pasta = (raiz / 'output/participacao_equipe_20261023').resolve()
    if not pasta.is_relative_to(raiz / 'output'):
        raise ValueError('Pasta de registros fora de output/.')
    arquivos = []
    for caminho in sorted(pasta.glob(participante + '_*.json')):
        if not caminho.resolve().is_relative_to(pasta):
            raise ValueError('Registro resolveu para fora da pasta de contribuições.')
        registro = json.loads(caminho.read_bytes())
        if registro.get('participante_id') != participante:
            raise ValueError('O participante do registro não corresponde ao nome do arquivo.')
        arquivos.append(caminho)
    if not arquivos:
        return None
    saida = (raiz / 'output/equipe_colab_20261023').resolve()
    if not saida.is_relative_to(raiz / 'output'):
        raise ValueError('Pasta de download fora de output/.')
    saida.mkdir(parents=True, exist_ok=True)
    agora = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    destino = saida / ('contribuicoes_' + participante + '_' + agora + '_' + uuid4().hex[:8] + '.zip')
    with ZipFile(destino, 'x', compression=ZIP_DEFLATED) as pacote:
        for caminho in arquivos:
            pacote.write(caminho, arcname=caminho.name)
        pacote.writestr('LEIA-ME.txt',
            'Contribuições autodeclaradas de ' + participante + '.\n'
            'Os JSONs preservam autoria informada, data, relato, evidências e uso de IA.\n'
            'Múltiplos arquivos podem representar versões sucessivas.\n'
            'A coordenação e o participante devem conferir os relatos; não há validação automática.\n')
    return destino

participante_download = "caio" # @param ["caio", "guilherme", "tiago", "rodolfo"]
baixar_contribuicoes = False # @param {type:"boolean"}
if baixar_contribuicoes:
    pacote_download = criar_pacote_contribuicoes(RAIZ, participante_download)
    if pacote_download is None:
        print('Nenhum registro encontrado. Preencha e execute seu formulário primeiro.')
    elif NO_COLAB:
        from google.colab import files
        files.download(str(pacote_download))
    else:
        print('Pacote salvo em:', pacote_download)
else:
    print('Depois de salvar seu formulário, selecione seu nome, marque a caixa e execute esta célula.')
